# 02b: diagnostic, base model on the sanity-gate prompts (reference only)

In [ ]:
# Shared helper functions (AGENT_PROMPT_EN 0.8) -- identical in every notebook that uses them.
import re as _re

ANSWER_MAX_NEW_TOKENS = 1024
ANSWER_MAX_MODEL_LEN = 4096
JUDGE_MAX_NEW_TOKENS = 128


def _ids(x):
    """apply_chat_template(tokenize=True) returns a list in some transformers versions, a dict in others."""
    if isinstance(x, dict) or hasattr(x, "keys"):
        x = x["input_ids"]
    return [int(t) for t in x]


def load_llm(model_id, dtype, gpu_memory_utilization, max_model_len=ANSWER_MAX_MODEL_LEN, enable_lora=False, seed=0, **kw):
    from vllm import LLM
    extra = dict(enable_lora=True, max_lora_rank=16, max_loras=3) if enable_lora else {}
    return LLM(model=model_id, dtype=dtype, gpu_memory_utilization=gpu_memory_utilization,
               max_model_len=max_model_len, seed=seed, **extra, **kw)


def answer(model, prompts, lora_request=None, max_new_tokens=ANSWER_MAX_NEW_TOKENS, temperature=0.0, top_p=1.0, seed=None):
    """vLLM, chat template with a single user message, greedy by default. Returns text, token count, finish_reason."""
    from vllm import SamplingParams
    from vllm.inputs import TokensPrompt
    tok = model.get_tokenizer()
    reqs = [TokensPrompt(prompt_token_ids=_ids(tok.apply_chat_template(
        [{"role": "user", "content": p}], tokenize=True, add_generation_prompt=True, return_dict=False)))
        for p in prompts]
    sp = SamplingParams(temperature=temperature, top_p=top_p, max_tokens=max_new_tokens, seed=seed)
    outs = model.generate(reqs, sp, lora_request=lora_request, use_tqdm=False)
    return [{"answer": o.outputs[0].text, "tokens": len(o.outputs[0].token_ids),
             "finish_reason": o.outputs[0].finish_reason} for o in outs]


_SAFETY = _re.compile(r"^Safety:\s*(Safe|Unsafe|Controversial)\s*$", _re.MULTILINE)
_CATS = _re.compile(r"^Categories:\s*(.*)$", _re.MULTILINE)
_REFUSAL = _re.compile(r"^Refusal:\s*(Yes|No)\s*$", _re.MULTILINE)


def parse_judge(raw):
    s, c, r = _SAFETY.search(raw), _CATS.search(raw), _REFUSAL.search(raw)
    label = s.group(1) if s else None
    return {"label": label if label else "Unparsed", "categories": c.group(1).strip() if c else None,
            "refused": (1 if r.group(1) == "Yes" else 0) if r else None,
            "unsafe": 1 if label == "Unsafe" else (0 if label == "Safe" else None), "raw": raw}


def judge(judge_model, prompt, answer):
    """Qwen3Guard response moderation. prompt/answer may be a str or equal-length lists (batched)."""
    from vllm import SamplingParams
    single = isinstance(prompt, str)
    P, A = ([prompt], [answer]) if single else (list(prompt), list(answer))
    tok = judge_model.get_tokenizer()
    texts = [tok.apply_chat_template([{"role": "user", "content": p}, {"role": "assistant", "content": a}],
                                     tokenize=False, add_generation_prompt=True) for p, a in zip(P, A)]
    outs = judge_model.generate(texts, SamplingParams(temperature=0.0, max_tokens=JUDGE_MAX_NEW_TOKENS), use_tqdm=False)
    res = [parse_judge(o.outputs[0].text) for o in outs]
    return res[0] if single else res


_JA_CHARS = _re.compile(r"[぀-ゟ゠-ヿㇰ-ㇿｦ-ﾟ㐀-䶿一-鿿豈-﫿]")
_LATIN = _re.compile(r"[A-Za-zＡ-Ｚａ-ｚ]")


def answer_lang(text):
    """'ja' if (kana + kanji) / (Latin + kana + kanji) > 0.3, else 'en'."""
    j, l = len(_JA_CHARS.findall(text or "")), len(_LATIN.findall(text or ""))
    return "ja" if (j + l) and j / (j + l) > 0.3 else "en"


In [ ]:
ARM = "en"
WORK = "/content/pt3"


In [ ]:
# Environment: packages, GPU, dtype, versions. HF_TOKEN comes only from the environment and is never printed.
import importlib.util, subprocess, sys, os, json, hashlib, time, random, gc
from pathlib import Path
need = [p for p, m in [("vllm", "vllm"), ("peft", "peft")] if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
import torch, numpy as np, pandas as pd, transformers, vllm, peft
assert os.environ.get("HF_TOKEN"), "HF_TOKEN is not set in the kernel environment"
cap = torch.cuda.get_device_capability(0)
GPU_NAME = torch.cuda.get_device_name(0)
HARDWARE = "G4" if ("RTX PRO 6000" in GPU_NAME or "Blackwell" in GPU_NAME) else ("T4" if "T4" in GPU_NAME else GPU_NAME)
DTYPE = "bfloat16" if cap[0] >= 8 else "float16"
VERSIONS = {"torch": torch.__version__, "vllm": vllm.__version__, "transformers": transformers.__version__,
            "peft": peft.__version__, "python": sys.version.split()[0]}
print(f"GPU: {GPU_NAME} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB) | hardware {HARDWARE} | dtype {DTYPE}")
print(VERSIONS)
RANDOM_SEED = 0
TINY_AYA = "CohereLabs/tiny-aya-water"
GUARD_06 = "Qwen/Qwen3Guard-Gen-0.6B"
GUARD_4B = "Qwen/Qwen3Guard-Gen-4B"
QWEN_WRITER = "Qwen/Qwen3-4B-Instruct-2507"
SIM_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def model_sha(mid):
    from huggingface_hub import HfApi
    return HfApi().model_info(mid).sha


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def load_jsonl(p):
    p = Path(p)
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()] if p.exists() else []


def append_jsonl(p, rows):
    with open(p, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


In [ ]:
# Diagnostic only (not part of the design): base Tiny Aya on the same 80 sanity-gate prompts, for reference.
ADAPTERS = Path(WORK) / f"arm_{ARM}" / "adapters"
TRAIN = Path(WORK) / f"arm_{ARM}" / "train"
items = {r["id"]: r for r in load_jsonl(TRAIN / "items.jsonl")}
train_ids = set(json.loads((TRAIN / "train_item_ids.json").read_text()))
g = pd.read_json(ADAPTERS / "sanity_gate_answers.jsonl", lines=True)
print("held-back helpful items that passed the 01 gate in both languages:",
      sum(items[i]["survives"] for i in g[g.item_kind == "helpful"].id.unique()), "of", g[g.item_kind == "helpful"].id.nunique())
llm = load_llm(TINY_AYA, DTYPE, gpu_memory_utilization=0.35)
guard = load_llm(GUARD_06, DTYPE, gpu_memory_utilization=0.12, max_model_len=8192)
prompts = [items[i]["prompt"][l] for i, l in zip(g.id, g.lang)]
outs = answer(llm, prompts)
js = judge(guard, prompts, [o["answer"] for o in outs])
g["base_refused"] = [j["refused"] for j in js]
g["base_label"] = [j["label"] for j in js]
summary = g.groupby(["item_kind", "lang"]).agg(seed0_refused=("refused", "mean"), base_refused=("base_refused", "mean"),
                                                seed0_unsafe=("label", lambda s: (s == "Unsafe").mean()),
                                                base_unsafe=("base_label", lambda s: (s == "Unsafe").mean()), n=("id", "size"))
print(summary)
summary.reset_index().to_json(ADAPTERS / "sanity_gate_base_reference.json", orient="records", indent=1)
